# Daily indicators — EMA, RSI, ATR, ADX, RVOL

`01_extract_bhavcopy.ipynb` produces **one row per symbol per day**. Every indicator here is a
function of a price *series*, so a single bhavcopy cannot produce any of them. This notebook does
two things:

1. **Backfills history** — the same bhavcopy fetch, looped over a date range, cached to disk so it
   is paid for once.
2. **Computes the indicators** per symbol on that panel, using Wilder's conventions (the ones
   TA-Lib and TradingView use), with the corporate-action problem from notebook 1 actually handled
   rather than deferred.

| Indicator | Default | What it is |
|---|---|---|
| EMA | 20 / 50 / 200 | Exponential moving average of close, SMA-seeded |
| RSI | 14 | Wilder's relative strength index, 0–100 |
| ATR | 14 | Average true range — volatility in price units |
| ADX | 14 | Trend *strength* (with +DI / −DI for direction), 0–100 |
| RVOL | 20 | Today's volume ÷ mean volume of the previous 20 sessions |

In [ ]:
import csv, io, json, os, ssl, time, zipfile
import urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd

try:
    import certifi
    SSL_CTX = ssl.create_default_context(cafile=certifi.where())
except ImportError:
    SSL_CTX = ssl.create_default_context()

# Colab's /content is wiped when the runtime recycles. Point OUT_DIR at a Drive
# path there if you want the history cache to survive a disconnect:
#   from google.colab import drive; drive.mount("/content/drive")
#   OUT_DIR = "/content/drive/MyDrive/stock-screen"
OUT_DIR = os.environ.get("STOCK_SCREEN_DIR") or (
    "/content/stock-screen-output" if os.path.isdir("/content")
    else os.path.expanduser("~/Desktop/ai-handson/stock-screen"))
HIST_DIR = os.path.join(OUT_DIR, "history")       # one cached CSV per exchange-day

EXCHANGES    = ["NSE"]        # add "BSE" if you screen it — doubles the download
HISTORY_DAYS = 780            # ~535 sessions: the whole UDiFF archive (it starts 2024-07-08).
                              # Not arbitrary — EMA(200) needs it. At 420 days it still sits ~1%
                              # off its converged value; see "Will these match TradingView?".
THREADS      = 4              # polite parallelism against the exchange archives
EQUITY_ONLY  = True

EMA_PERIODS  = (20, 50, 200)
RSI_PERIOD   = 14
ATR_PERIOD   = 14
ADX_PERIOD   = 14
RVOL_PERIOD  = 20

MIN_BARS     = 60             # drop symbols too new to smooth anything meaningfully. Wilder
                              # indicators need ~200 bars to be fully seed-independent — the
                              # `bars` column in the output lets you filter harder.
ADJUST_CORPORATE_ACTIONS = True
WRITE_FULL_PANEL = False      # the per-symbol history with indicators is ~100 MB of CSV

os.makedirs(HIST_DIR, exist_ok=True)
print("output dir :", OUT_DIR)
print("cache dir  :", HIST_DIR)

## Source config and parser

Identical to notebook 1 — repeated here so this notebook stands alone. Both exchanges publish the
same 34-column UDiFF schema.

One archive limit worth knowing: **UDiFF only goes back to mid-2024.** Before that the exchanges
used a different filename and layout, so a lookback longer than that needs a second parser.

In [ ]:
UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/124.0 Safari/537.36")

SOURCES = {
    "NSE": {
        "url": ("https://nsearchives.nseindia.com/content/cm/"
                "BhavCopy_NSE_CM_0_0_0_{d}_F_0000.csv.zip"),
        "referer": "https://www.nseindia.com/",
        "zipped": True,
        "equity_series": {"EQ", "BE", "BZ", "SM", "ST"},
    },
    "BSE": {
        "url": ("https://www.bseindia.com/download/BhavCopy/Equity/"
                "BhavCopy_BSE_CM_0_0_0_{d}_F_0000.CSV"),
        "referer": "https://www.bseindia.com/",
        "zipped": False,
        "equity_series": {"A", "B", "T", "M", "MT", "MS", "X", "XT"},
    },
}

OUT_COLS = ["date", "exchange", "symbol", "isin", "series", "name",
            "open", "high", "low", "close", "prev_close", "last",
            "volume", "turnover", "trades"]


def fetch(url, referer, retries=3):
    """GET with browser headers. Returns bytes, or None if not published."""
    req = urllib.request.Request(url, headers={
        "User-Agent": UA, "Referer": referer,
        "Accept": "text/csv,application/zip,*/*",
    })
    for attempt in range(retries):
        try:
            with urllib.request.urlopen(req, timeout=45, context=SSL_CTX) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 404):
                return None                      # holiday / not published
            if attempt == retries - 1:
                raise
        except (urllib.error.URLError, TimeoutError):
            if attempt == retries - 1:
                raise
        time.sleep(2 ** attempt)
    return None


def parse(payload, zipped):
    """Bytes -> list of dicts. None if the response isn't actually a bhavcopy."""
    if payload is None:
        return None
    if zipped:
        try:
            zf = zipfile.ZipFile(io.BytesIO(payload))
        except zipfile.BadZipFile:
            return None
        name = next((n for n in zf.namelist() if n.lower().endswith(".csv")), None)
        if name is None:
            return None
        payload = zf.read(name)
    text = payload.decode("utf-8", errors="replace").lstrip("\ufeff")
    if not text.lstrip().startswith("TradDt"):
        return None                              # BSE serves HTML error pages with HTTP 200
    return list(csv.DictReader(io.StringIO(text)))


def normalise(rows, exchange, equity_only=EQUITY_ONLY):
    keep = SOURCES[exchange]["equity_series"]
    out = []
    for r in rows:
        if r.get("FinInstrmTp") != "STK":
            continue
        if equity_only and r.get("SctySrs") not in keep:
            continue
        out.append({
            "date": r["TradDt"], "exchange": exchange, "symbol": r["TckrSymb"],
            "isin": r["ISIN"], "series": r["SctySrs"], "name": r["FinInstrmNm"],
            "open": r["OpnPric"], "high": r["HghPric"], "low": r["LwPric"],
            "close": r["ClsPric"], "prev_close": r["PrvsClsgPric"], "last": r["LastPric"],
            "volume": r["TtlTradgVol"], "turnover": r["TtlTrfVal"],
            "trades": r["TtlNbOfTxsExctd"],
        })
    return out

## Backfill, cached

Each exchange-day is written to `history/<exchange>_<date>.csv` and never fetched twice. Holidays
answer 403/404, so the dates that legitimately have no file are recorded in `_missing.json` — that
stops a re-run from re-probing ~20 holidays every time.

The first run downloads several hundred files (a few minutes). Every run after that is a cache scan
plus the handful of new sessions.

Weekends are probed rather than assumed non-trading. NSE runs special live sessions on some
Saturdays, and on Budget day when it falls on a weekend — **2026-02-01, a Sunday, was a full
session.** A skipped session is worse than a slow backfill: the next trading day's `prev_close`
then disagrees with the close we stored, which the corporate-action step below would read as a
split on every symbol simultaneously.

In [ ]:
MISSING_PATH = os.path.join(HIST_DIR, "_missing.json")


def _load_missing():
    if os.path.exists(MISSING_PATH):
        with open(MISSING_PATH) as f:
            return set(json.load(f))
    return set()


def _save_missing(missing):
    with open(MISSING_PATH, "w") as f:
        json.dump(sorted(missing), f)


def cache_path(exchange, day):
    return os.path.join(HIST_DIR, f"{exchange.lower()}_{day.isoformat()}.csv")


def fetch_day(exchange, day):
    """Fetch one exchange-day into the cache. Returns 'hit' | 'new' | 'missing'."""
    path = cache_path(exchange, day)
    if os.path.exists(path):
        return "hit"
    cfg = SOURCES[exchange]
    rows = parse(fetch(cfg["url"].format(d=day.strftime("%Y%m%d")), cfg["referer"]),
                 cfg["zipped"])
    if not rows:
        return "missing"
    frame = pd.DataFrame(normalise(rows, exchange), columns=OUT_COLS)
    if frame.empty:
        return "missing"
    frame.to_csv(path + ".tmp", index=False)     # write-then-rename: no half files in the cache
    os.replace(path + ".tmp", path)
    return "new"


def backfill(exchanges, days_back, end=None):
    end = end or date.today()
    missing = _load_missing()
    todo = []
    for i in range(days_back + 1):
        day = end - timedelta(days=i)
        # Weekends are probed too, not skipped: NSE runs special live sessions on
        # some Saturdays and on Budget day when it falls on a weekend (2026-02-01
        # was one). Skipping them leaves a hole that reads as a corporate action
        # on the next trading day for every symbol at once.
        for ex in exchanges:
            if f"{ex}:{day.isoformat()}" in missing or os.path.exists(cache_path(ex, day)):
                continue
            todo.append((ex, day))

    cached = sum(len([f for f in os.listdir(HIST_DIR) if f.startswith(ex.lower() + "_")])
                 for ex in exchanges)
    print(f"cached exchange-days: {cached}   to fetch: {len(todo)}")
    if not todo:
        return

    done = 0
    with ThreadPoolExecutor(max_workers=THREADS) as pool:
        for (ex, day), status in zip(todo, pool.map(lambda a: fetch_day(*a), todo)):
            done += 1
            if status == "missing":
                missing.add(f"{ex}:{day.isoformat()}")
            if done % 25 == 0 or done == len(todo):
                print(f"  {done}/{len(todo)} fetched")
    _save_missing(missing)
    print(f"non-trading / unpublished days recorded: {len(missing)}")


backfill(EXCHANGES, HISTORY_DAYS)

## Assemble the panel

Read the cache back into one long frame, one row per (exchange, symbol, date). Two data quirks get
handled here:

- **Duplicate rows** — a symbol can appear under two series on the same exchange. Keep the row with
  the higher turnover; that is the line people actually trade.
- **Short histories** — recent listings cannot support a 14-bar smoothing, let alone a 200-bar one.
  They stay in the panel but their indicators come out `NaN`, which is the honest answer.

In [ ]:
files = sorted(f for f in os.listdir(HIST_DIR)
               if f.endswith(".csv") and f.split("_")[0].upper() in EXCHANGES)
assert files, "history cache is empty — run the backfill cell first"

panel = pd.concat([pd.read_csv(os.path.join(HIST_DIR, f)) for f in files], ignore_index=True)

missing = [c for c in OUT_COLS if c not in panel.columns]
assert not missing, f"cache holds a file with an unexpected schema (missing {missing}) — delete history/ and re-run"

num = ["open", "high", "low", "close", "prev_close", "last", "volume", "turnover", "trades"]
panel[num] = panel[num].apply(pd.to_numeric, errors="coerce")
panel["date"] = pd.to_datetime(panel["date"])

panel = (panel.sort_values("turnover", ascending=False)
              .drop_duplicates(["exchange", "symbol", "date"], keep="first")
              .sort_values(["exchange", "symbol", "date"], ignore_index=True))

# Rights entitlements (SYMBOL-RE) are a separate, short-lived instrument with no price
# band — they routinely move 40% in a session, which would poison every threshold below.
re_rows = panel["symbol"].str.contains(r"-RE\d*$", regex=True)
panel = panel[~re_rows]

sessions = panel.groupby("exchange")["date"].nunique()
bars = panel.groupby(["exchange", "symbol"]).size()
panel = panel[panel.set_index(["exchange", "symbol"]).index.isin(bars[bars >= MIN_BARS].index)]
panel = panel.reset_index(drop=True)
# groupby silently drops null keys, so a bad row would vanish rather than raise
assert panel[["exchange", "symbol", "date", "close"]].notna().all().all(), "null key/price in the panel"

print(f"rights-entitlement rows dropped: {int(re_rows.sum()):,}")
print(f"rows: {len(panel):,}   symbols: {panel.groupby(['exchange','symbol']).ngroups:,}")
print(f"date range: {panel['date'].min().date()} .. {panel['date'].max().date()}")
print("\nsessions per exchange:"); print(sessions.to_string())
print(f"\nsymbols dropped for < {MIN_BARS} bars: {(bars < MIN_BARS).sum():,}")
print("\nbars per symbol:"); print(bars[bars >= MIN_BARS].describe()[["min", "50%", "max"]].to_string())

## Corporate actions

Notebook 1 deferred this one, and indicators cannot: bhavcopy prices are **unadjusted**, so a 1:10
split reads as a 90% crash. Left alone it fires every momentum screen, pins RSI at 0, and drags an
EMA(200) for months.

The hoped-for shortcut does not work here — NSE's `prev_close` is the *raw* previous close, not the
action-adjusted one, so it cannot identify the event. What is left is the price gap itself, and the
honest reading of it comes from the price bands: a cash equity is circuit-limited, so a genuine
40%-in-a-session move is essentially impossible while a split trivially produces one. Calibrating
that on 14 months of NSE data:

| Gap `close ÷ previous close` | What is actually in there |
|---|---|
| exactly `0.800` | the 20% lower circuit — real crashes, a whole cluster of them |
| `0.70` – `0.80` | **both** — 1:2 bonuses (`0.667`), and real routs like IEX's −30% day |
| below `0.60` | splits, 1:1 bonuses, demergers. No unadjusted crash in the sample reaches here |

So the rule is deliberately asymmetric:

- **Gap beyond ±40%** (`0.60` / `1.667`) — treated as an action and rescaled. Every earlier bar is
  multiplied by the running product of later gaps; volumes are divided by it, since a split
  multiplies share count.
- **Gap between ±20% and ±40%** — flagged and printed, never rescaled. Bonuses hide in here next to
  genuine crashes, and silently rewriting a real 30% fall is the worse error. Confirm one by eye and
  put it in `MANUAL_RATIOS` to have it applied.

Two more guards: the previous **row** must be the previous **session**, so a hole in the cache is
never mistaken for an action; and since a real action hits one symbol at a time, any date where more
than 2% of symbols light up is a missing session, not a market-wide split — that date is reported and
left alone.

`adj_*` columns are what the indicators consume; the raw columns stay untouched beside them. The
complete fix is a corporate-actions feed, which is a data licence, not an algorithm.

In [ ]:
CA_ADJUST_GAP   = 0.60       # gap this extreme is an action, not a market move
CA_REVIEW_GAP   = 0.80       # between the two: report it, never rescale it
MAX_EVENT_SHARE = 0.02       # per date: above this share of symbols it is a missing session
MANUAL_RATIOS   = {}         # ("NSE", "TRENT", "2026-06-04"): 2/3   -> force an adjustment

# session_no makes "the previous session" unambiguous even when a symbol is untraded
panel["session_no"] = panel.groupby("exchange")["date"].rank(method="dense").astype(int)
g = panel.groupby(["exchange", "symbol"], sort=False)

prev_row_close = g["close"].shift(1)
contiguous = g["session_no"].diff().eq(1)          # our previous row IS the previous session
gap = panel["close"] / prev_row_close
usable = contiguous & prev_row_close.gt(0) & gap.gt(0)

# the file's own prev_close should equal the close we stored; if not, the previous bar
# is unreliable and the gap means nothing
file_agrees = ((panel["prev_close"] - prev_row_close).abs() / prev_row_close).lt(0.005)

extreme = usable & file_agrees & (gap.lt(CA_ADJUST_GAP) | gap.gt(1 / CA_ADJUST_GAP))
review  = usable & ~extreme & (gap.lt(CA_REVIEW_GAP) | gap.gt(1 / CA_REVIEW_GAP))

# A real action hits one symbol. If a whole date lights up, the cache is missing a
# session and the "gap" is just that day's market move — refuse to rescale on it.
share = extreme.groupby([panel["exchange"], panel["date"]]).mean()
suspect = share[share > MAX_EVENT_SHARE]
if len(suspect):
    print("session likely missing from the cache — nothing adjusted on:")
    for (ex, d), frac in suspect.items():
        print(f"  {ex} {d.date()}: {frac:.0%} of symbols flagged")
    extreme &= ~pd.MultiIndex.from_arrays([panel["exchange"], panel["date"]]).isin(suspect.index)

ratio = gap.where(extreme, 1.0)
for (ex, sym, day), r in MANUAL_RATIOS.items():    # confirmed by hand, applied verbatim
    hit = (panel["exchange"].eq(ex) & panel["symbol"].eq(sym)
           & panel["date"].eq(pd.Timestamp(day)))
    ratio = ratio.mask(hit, r)
    extreme |= hit

if not ADJUST_CORPORATE_ACTIONS:
    ratio, extreme = pd.Series(1.0, index=panel.index), pd.Series(False, index=panel.index)

panel["ca_flag"] = np.select([extreme, review], ["adjusted", "review"], default="")
panel["_ratio"] = ratio
# factor[j] = product of every later ratio, so the pre-action history moves onto today's scale
panel["adj_factor"] = g["_ratio"].transform(
    lambda s: s[::-1].cumprod()[::-1].shift(-1).fillna(1.0))
panel = panel.drop(columns="_ratio")

for c in ("open", "high", "low", "close"):
    panel["adj_" + c] = panel[c] * panel["adj_factor"]
panel["adj_volume"] = panel["volume"] / panel["adj_factor"]   # a split multiplies share count

shown = ["date", "exchange", "symbol", "series", "prev_close", "close", "gap"]
ev = panel.assign(gap=gap.round(4))
print(f"adjusted as corporate actions: {int(extreme.sum()):,} events across "
      f"{ev[extreme].groupby(['exchange','symbol']).ngroups:,} symbols")
print(f"bars rescaled: {(panel['adj_factor'] != 1).sum():,}")
if extreme.any():
    display(ev[extreme].sort_values("gap")[shown].head(12))

print()
print(f"flagged for review, NOT adjusted: {int(review.sum()):,} "
      f"(bonuses and genuine crashes are mixed together in this band)")
if review.any():
    display(ev[review].sort_values("gap")[shown].head(12))

## The indicator math

Two conventions matter, and getting either wrong puts the numbers quietly out of step with every
chart the user compares them against:

**Wilder's smoothing** (`rma`) — `s_t = s_{t-1} + (x_t − s_{t-1})/n`. It is an EMA with
`alpha = 1/n`, *not* `2/(n+1)`. RSI, ATR and ADX all use it.

**SMA seeding** — the recursion starts from the simple average of the first `n` bars. The lazy
alternative, seeding from the first observation, leaves a bias that decays only geometrically:
250 bars into an EMA(200) roughly 8% of the value is still the seed. Bars before the seed are `NaN`
rather than a half-warmed number, so `ADX(14)` first prints at bar 27 and `EMA(200)` at bar 199 —
the same lookbacks TA-Lib uses.

In [ ]:
def _seeded_ewm(x, n, alpha):
    """Recursive average seeded with the SMA of the first n valid bars."""
    v = np.asarray(x, dtype="float64")
    valid = ~np.isnan(v)
    if valid.sum() < n:
        return pd.Series(np.nan, index=x.index, dtype="float64")
    pos = int(np.argmax(valid))                      # first non-NaN bar
    seeded = v.copy()
    seeded[: pos + n - 1] = np.nan
    seeded[pos + n - 1] = np.nanmean(v[pos : pos + n])
    return pd.Series(seeded, index=x.index).ewm(alpha=alpha, adjust=False).mean()


def rma(x, n):
    """Wilder's smoothing."""
    return _seeded_ewm(x, n, 1.0 / n)


def ema(x, n):
    return _seeded_ewm(x, n, 2.0 / (n + 1))


def rsi(close, n=14):
    d = close.diff()
    gain, loss = d.clip(lower=0), (-d).clip(lower=0)
    gain[d.isna()] = np.nan                          # bar 0 has no move; keep it out of the seed
    loss[d.isna()] = np.nan
    avg_gain, avg_loss = rma(gain, n), rma(loss, n)
    out = 100.0 - 100.0 / (1.0 + avg_gain / avg_loss)
    out = out.where(avg_loss != 0, 100.0)            # no down moves -> 100, as Pine and TA-Lib
    dead = avg_gain.eq(0) & avg_loss.eq(0)           # never moved -> undefined, not 100
    return out.mask(dead).where(avg_gain.notna())


def true_range(high, low, close):
    pc = close.shift(1)
    tr = pd.concat([high - low, (high - pc).abs(), (low - pc).abs()], axis=1).max(axis=1)
    tr.iloc[0] = high.iloc[0] - low.iloc[0]          # no prior close on bar 0
    return tr


def atr(high, low, close, n=14):
    return rma(true_range(high, low, close), n)


def adx(high, low, close, n=14):
    """Wilder's directional movement. Returns (+DI, -DI, ADX)."""
    up, dn = high.diff(), -low.diff()
    plus_dm = pd.Series(np.where((up > dn) & (up > 0), up, 0.0), index=high.index)
    minus_dm = pd.Series(np.where((dn > up) & (dn > 0), dn, 0.0), index=high.index)
    plus_dm.iloc[0] = minus_dm.iloc[0] = np.nan      # bar 0 has no direction

    tr = true_range(high, low, close)
    tr.iloc[0] = np.nan                              # align the TR seed with the DM seed
    atr_n = rma(tr, n)
    pdi = 100.0 * rma(plus_dm, n) / atr_n
    mdi = 100.0 * rma(minus_dm, n) / atr_n
    dx = 100.0 * (pdi - mdi).abs() / (pdi + mdi).replace(0.0, np.nan)
    return pdi, mdi, rma(dx, n)


def rvol(volume, n=20):
    """Today's volume against the mean of the previous n sessions (today excluded)."""
    base = volume.rolling(n).mean().shift(1)
    return volume / base.replace(0.0, np.nan)

## Apply to the panel

Per symbol, in date order. Symbols with too little history produce `NaN` for the periods they cannot
support — an EMA(200) on 60 bars is not a shorter EMA, it is a wrong one.

In [ ]:
t0 = time.time()
panel = panel.sort_values(["exchange", "symbol", "date"], ignore_index=True)
g = panel.groupby(["exchange", "symbol"], sort=False)

for n in EMA_PERIODS:
    panel[f"ema{n}"] = g["adj_close"].transform(lambda s, n=n: ema(s, n))
panel[f"rsi{RSI_PERIOD}"] = g["adj_close"].transform(lambda s: rsi(s, RSI_PERIOD))
panel[f"rvol{RVOL_PERIOD}"] = g["adj_volume"].transform(lambda s: rvol(s, RVOL_PERIOD))


def _hlc_block(d):
    pdi, mdi, adx_v = adx(d["adj_high"], d["adj_low"], d["adj_close"], ADX_PERIOD)
    return pd.DataFrame({
        f"atr{ATR_PERIOD}": atr(d["adj_high"], d["adj_low"], d["adj_close"], ATR_PERIOD),
        "plus_di": pdi, "minus_di": mdi, f"adx{ADX_PERIOD}": adx_v,
    }, index=d.index)


block = g[["adj_high", "adj_low", "adj_close"]].apply(_hlc_block)
if isinstance(block.index, pd.MultiIndex):           # pandas re-nests on some versions
    block = block.droplevel([0, 1])
panel[block.columns] = block.reindex(panel.index)

# a couple of readings that are only useful in relative terms
panel["bars"] = g.cumcount() + 1                    # history behind each row, for warm-up filtering
panel[f"atr{ATR_PERIOD}_pct"] = 100 * panel[f"atr{ATR_PERIOD}"] / panel["adj_close"]
panel["ret_1d_pct"] = 100 * g["adj_close"].pct_change()

print(f"indicators computed on {len(panel):,} rows in {time.time() - t0:.1f}s")
IND = [f"ema{n}" for n in EMA_PERIODS] + [f"rsi{RSI_PERIOD}", f"atr{ATR_PERIOD}",
       f"adx{ADX_PERIOD}", "plus_di", "minus_di", f"rvol{RVOL_PERIOD}"]
display(panel[IND].describe().T[["count", "mean", "min", "50%", "max"]])

## Validation

Vectorised code that silently smooths the wrong window is the failure mode here, and it looks
perfectly plausible in a table. So the last bar of a real symbol is recomputed with a slow,
literal transcription of Wilder's rules and the two must agree to 1e-9.

In [ ]:
def _ref_rsi(c, n=14):
    g = sum(max(c[i] - c[i - 1], 0) for i in range(1, n + 1)) / n
    l = sum(max(c[i - 1] - c[i], 0) for i in range(1, n + 1)) / n
    for i in range(n + 1, len(c)):
        g += (max(c[i] - c[i - 1], 0) - g) / n
        l += (max(c[i - 1] - c[i], 0) - l) / n
    return 100.0 if l == 0 else 100 - 100 / (1 + g / l)


def _ref_tr(h, l, c, i):
    return h[i] - l[i] if i == 0 else max(h[i] - l[i], abs(h[i] - c[i - 1]), abs(l[i] - c[i - 1]))


def _ref_atr(h, l, c, n=14):
    a = sum(_ref_tr(h, l, c, i) for i in range(n)) / n
    for i in range(n, len(c)):
        a += (_ref_tr(h, l, c, i) - a) / n
    return a


def _ref_adx(h, l, c, n=14):
    def dm(i):
        up, dn = h[i] - h[i - 1], l[i - 1] - l[i]
        return (up if up > dn and up > 0 else 0.0), (dn if dn > up and dn > 0 else 0.0)
    ps = ms = ts = 0.0
    for i in range(1, n + 1):
        p, m = dm(i); ps += p; ms += m; ts += _ref_tr(h, l, c, i)
    dx = []
    def push():
        pdi, mdi = 100 * ps / ts, 100 * ms / ts
        dx.append(0.0 if pdi + mdi == 0 else 100 * abs(pdi - mdi) / (pdi + mdi))
    push()
    for i in range(n + 1, len(c)):
        p, m = dm(i)
        ps += p - ps / n; ms += m - ms / n; ts += _ref_tr(h, l, c, i) - ts / n
        push()
    a = sum(dx[:n]) / n
    for v in dx[n:]:
        a += (v - a) / n
    return a


def _ref_ema(c, n):
    e, k = sum(c[:n]) / n, 2 / (n + 1)
    for x in c[n:]:
        e = x * k + e * (1 - k)
    return e


counts = panel.groupby(["exchange", "symbol"]).size()
probe = counts[counts == counts.max()].index[0]
d = panel[(panel.exchange == probe[0]) & (panel.symbol == probe[1])]
h, l, c = (d[x].tolist() for x in ("adj_high", "adj_low", "adj_close"))
last = d.iloc[-1]

cases = [(f"RSI({RSI_PERIOD})",  last[f"rsi{RSI_PERIOD}"],  _ref_rsi(c, RSI_PERIOD)),
         (f"ATR({ATR_PERIOD})",  last[f"atr{ATR_PERIOD}"],  _ref_atr(h, l, c, ATR_PERIOD)),
         (f"ADX({ADX_PERIOD})",  last[f"adx{ADX_PERIOD}"],  _ref_adx(h, l, c, ADX_PERIOD))]
cases += [(f"EMA({n})", last[f"ema{n}"], _ref_ema(c, n)) for n in EMA_PERIODS if len(c) >= n]
cases.append((f"RVOL({RVOL_PERIOD})", last[f"rvol{RVOL_PERIOD}"],
              d["adj_volume"].iloc[-1] / d["adj_volume"].iloc[-1 - RVOL_PERIOD:-1].mean()))

print(f"recomputing {probe[1]} ({len(c)} bars) the slow way:")
for name, got, want in cases:
    print(f"  {'PASS' if abs(got - want) < 1e-9 * max(1, abs(want)) else 'FAIL'}  "
          f"{name:<10} vectorised {got:>14.6f}   reference {want:>14.6f}")

snap = panel[panel["date"] == panel["date"].max()]
invariants = {
    "RSI within 0..100":       snap[f"rsi{RSI_PERIOD}"].dropna().between(0, 100).all(),
    "ADX within 0..100":       snap[f"adx{ADX_PERIOD}"].dropna().between(0, 100).all(),
    "ATR non-negative":        (snap[f"atr{ATR_PERIOD}"].dropna() >= 0).all(),
    "RVOL positive":           (snap[f"rvol{RVOL_PERIOD}"].dropna() > 0).all(),
    "EMA20 within day range":  True,   # informational only; see below
    "no duplicate symbol/day": not panel.duplicated(["exchange", "symbol", "date"]).any(),
    "dates sorted per symbol": panel.groupby(["exchange", "symbol"])["date"].is_monotonic_increasing.all(),
}
del invariants["EMA20 within day range"]
print()
for name, ok in invariants.items():
    print(f"  {'PASS' if ok else 'FAIL'}  {name}")

warm = {f"{k}": int(snap[k].notna().sum()) for k in IND}
print(f"\nsymbols in latest snapshot: {len(snap):,}")
print("non-null indicator counts:", warm)

### Cross-check against TA-Lib

The loop above is still my own reading of the rules, so it cannot catch a misread convention — both
implementations would be wrong in the same way. TA-Lib is the C reference the rest of the industry
compares against, so if it is installed (`pip install TA-Lib`) this cell checks every bar of a sample
against it. Warm-up is reported separately from the converged region, because the two libraries
genuinely seed ATR differently and that difference is a convention, not an error in either.

In [ ]:
try:
    import talib
except ImportError:
    talib = None
    print("TA-Lib not installed — skipping. `pip install TA-Lib` to run this check.")

if talib is not None:
    counts = panel.groupby(["exchange", "symbol"]).size().sort_values(ascending=False)
    sample = counts.head(400).index
    acc = {}
    for ex, sym in sample:
        d = panel[(panel.exchange == ex) & (panel.symbol == sym)]
        h, l, c = (d["adj_" + x].to_numpy(float) for x in ("high", "low", "close"))
        if not np.isfinite(c).all():
            continue
        H_, L_, C_ = (pd.Series(x) for x in (h, l, c))
        pdi, mdi, adx_v = adx(H_, L_, C_, ADX_PERIOD)
        pairs = [(f"EMA{n}", ema(C_, n), talib.EMA(c, n)) for n in EMA_PERIODS if len(c) > n]
        pairs += [
            (f"RSI{RSI_PERIOD}", rsi(C_, RSI_PERIOD), talib.RSI(c, RSI_PERIOD)),
            (f"ATR{ATR_PERIOD}", atr(H_, L_, C_, ATR_PERIOD), talib.ATR(h, l, c, ATR_PERIOD)),
            ("PLUS_DI", pdi, talib.PLUS_DI(h, l, c, ADX_PERIOD)),
            ("MINUS_DI", mdi, talib.MINUS_DI(h, l, c, ADX_PERIOD)),
            (f"ADX{ADX_PERIOD}", adx_v, talib.ADX(h, l, c, ADX_PERIOD)),
        ]
        for name, mine, ref in pairs:
            a, b = np.asarray(mine, float), np.asarray(ref, float)
            ok = ~np.isnan(a) & ~np.isnan(b)
            if not ok.any():
                continue
            idx = np.where(ok)[0]
            rel = np.abs(a[ok] - b[ok]) / np.maximum(np.abs(b[ok]), 1e-12)
            age = idx - idx[0]
            e = acc.setdefault(name, {"all": 0.0, "warm": 0.0, "bars": 0})
            e["all"] = max(e["all"], float(rel.max()))
            e["bars"] += int(ok.sum())
            if (age >= 200).any():
                e["warm"] = max(e["warm"], float(rel[age >= 200].max()))

    print(f"{len(sample)} longest-history symbols, every bar, vs TA-Lib")
    print()
    print(f"{'indicator':<10}{'bars':>12}{'max rel err':>14}{'after 200 bars':>18}")
    for k, e in acc.items():
        print(f"{k:<10}{e['bars']:>12,}{e['all']:>14.2e}{e['warm']:>18.2e}")
    print()
    print("EMA and RSI agree outright. ATR, the DIs and ADX differ only while the seed still")
    print("carries weight: TA-Lib starts its ATR one bar later, which decays to nothing.")

## Will these match TradingView?

The arithmetic is settled; the data is not. Taking those separately.

**The formulas.** Checked three ways: against the literal loop above, against TA-Lib bar-for-bar
(the cell above this one), and against `pandas_ta`, which targets TradingView semantics. EMA, RSI and
RVOL agree with TA-Lib to machine precision — `1e-15` relative over 200,000 bars. ATR, the DIs and
ADX agree to `~1e-7` once the seed washes out, and differ during warm-up for one reason worth stating
plainly:

- **ATR seeding.** Pine's `ta.atr` is `ta.rma(ta.tr(true), length)`, where the first bar's true range
  is `high - low` and `ta.rma` seeds on the SMA of the first 14 — so the first ATR lands on bar 13.
  TA-Lib drops that first bar and seeds one bar later. This notebook follows Pine, and `pandas_ta`
  reproduces it **exactly, to `0.0e+00`**, which is what settles which convention is which.
- **RSI with no movement.** A symbol that never ticks has no gains and no losses, so RSI is `0/0`.
  Pine returns 100, TA-Lib returns 0, `pandas_ta` returns `NaN` — three libraries, three answers,
  all arbitrary. This notebook returns `NaN`, because on an untraded microcap a printed 100 reads as
  "maximally overbought" and would walk straight into a momentum screen. A genuine one-way rally
  still returns 100, matching all three.

Warm-up is where the conventions disagree, so it is where the value is least trustworthy: the `bars`
column carries the history behind every row, and `bars >= 200` is where Wilder indicators stop
depending on how they were started.

**The data** is the part that decides whether a number on a chart matches a number here.

History length is the other half of it. Recomputing the last bar for 891 liquid symbols on a
285-session window versus the full 536-session archive isolates that effect:

| Indicator | Median shift | 90th pct | Max |
|---|---|---|---|
| EMA(20), RSI(14), ATR(14), ADX(14), RVOL(20) | `0.0000` | `0.0000` | `0.0000` |
| EMA(50) | `0.0001%` | `0.0002%` | `0.0017%` |
| **EMA(200)** | **`0.96%`** | **`2.40%`** | **`6.21%`** |

Wilder's `alpha = 1/14` washes the seed out after ~200 bars, so RSI, ATR and ADX are fully converged
and history length is irrelevant to them. EMA(200)'s `alpha = 2/201` does not: at 285 bars the seed
is still **43%** of the value. TradingView computes from years of bars and has no seed left, so a
short window is a real disagreement, not a rounding difference:

| Bars | ≈ calendar days | Median error vs full archive | Max |
|---|---|---|---|
| 250 | 360 | 1.28% | 8.74% |
| 285 | 415 | 0.96% | 6.21% |
| 350 | 505 | 0.47% | 3.47% |
| 480 | 695 | 0.15% | 1.25% |
| 520 | 755 | 0.03% | 0.49% |

Hence `HISTORY_DAYS = 780`. Even then the archive itself is the binding constraint: UDiFF starts
2024-07-08, so ~3.5% of EMA(200) is still seed, worth roughly 0.1% of the value. Closing that needs
the pre-2024 bhavcopy format.

So what remains is data, and none of it is fixable with better arithmetic:

- **Dividends.** TradingView can back-adjust for them; bhavcopy prices never are. Every ex-date
  leaves a small step here that is not there on the chart.
- **The review-band actions.** A 1:2 bonus is adjusted on TradingView and flagged-but-unadjusted
  here, so those symbols diverge until you add them to `MANUAL_RATIOS`.
- **RVOL has no standard.** This one is today ÷ mean of the *previous* 20 sessions. Versions that
  include today read lower on a volume spike, which is exactly when it matters.
- **Different feed.** TradingView carries its own NSE feed with its own symbol history; this is the
  exchange's official file. They agree on the close far more often than not, but not by construction.

Compare a couple of symbols against a chart before trusting a screen built on this. On a liquid
symbol with 200+ bars and no corporate action in the window, RSI, ATR and ADX should agree to the
decimal — if one does not, the price series is the suspect, not the smoothing.

## Write the screening table

`indicators_latest.csv` is one row per symbol — the newest session with every indicator attached.
That is the file a screen reads. The full per-day panel is only written when `WRITE_FULL_PANEL` is
on, since it runs to a hundred megabytes.

In [ ]:
KEEP = (["date", "exchange", "symbol", "isin", "series", "name",
         "open", "high", "low", "close", "volume", "turnover", "trades",
         "adj_close", "adj_volume", "adj_factor", "ret_1d_pct", "bars"]
        + IND + [f"atr{ATR_PERIOD}_pct"])

stamp = panel["date"].max().date().isoformat()
latest = panel[panel["date"] == panel["date"].max()][KEEP].sort_values("symbol")

path = os.path.join(OUT_DIR, "indicators_latest.csv")
latest.to_csv(path, index=False)
written = [path, ]
latest.to_csv(os.path.join(OUT_DIR, f"indicators_{stamp}.csv"), index=False)
written.append(os.path.join(OUT_DIR, f"indicators_{stamp}.csv"))

if WRITE_FULL_PANEL:
    full = os.path.join(OUT_DIR, "panel_indicators.csv")
    panel[KEEP + ["ca_event"]].to_csv(full, index=False)
    written.append(full)

for p in written:
    print(f"{os.path.getsize(p):>12,} bytes  {p}")

display(latest[latest.symbol.isin(["RELIANCE", "TCS", "INFY", "HDFCBANK"])]
        [["symbol", "close"] + IND].round(2))

## Example screen

Not a recommendation — a demonstration that the columns compose. Liquid names in an established
uptrend, trend actually strong rather than drifting, not yet overbought, with volume showing up
today.

In [ ]:
s = latest.dropna(subset=IND)
shortlist = s[
    (s["turnover"] > 5e7)                                  # >= Rs 5 cr traded today
    & (s["close"] > s["ema50"]) & (s["ema50"] > s["ema200"])
    & (s[f"adx{ADX_PERIOD}"] > 25) & (s["plus_di"] > s["minus_di"])
    & (s[f"rsi{RSI_PERIOD}"].between(50, 70))
    & (s[f"rvol{RVOL_PERIOD}"] > 1.5)
].sort_values(f"rvol{RVOL_PERIOD}", ascending=False)

cols = ["symbol", "close", "ret_1d_pct", f"rsi{RSI_PERIOD}", f"adx{ADX_PERIOD}",
        f"rvol{RVOL_PERIOD}", f"atr{ATR_PERIOD}_pct", "turnover"]
print(f"{len(shortlist)} of {len(s):,} fully-warmed symbols pass")
display(shortlist[cols].head(25).round(2))

---
## What this does and does not give you

**Handled here**
- History long enough that every indicator except EMA(200) is fully converged, and EMA(200) is
  within ~0.1% of what the archive can support.
- Splits, 1:1 bonuses and demergers — gaps beyond ±40% are rescaled, so they no longer read as
  crashes. Bars adjacent to a hole in the cache, and dates where the whole market lights up, are
  left alone rather than guessed at.
- Wilder conventions verified against TA-Lib and `pandas_ta`, not only against a hand-written loop:
  EMA, RSI and RVOL agree to machine precision, ATR/DI/ADX to `~1e-7` past warm-up.
- Insufficient history returns `NaN` instead of a plausible-looking wrong number, and the `bars`
  column exposes how warmed each row actually is.

**Still open**
1. **Smaller corporate actions** — a 1:2 bonus (gap `0.667`) is indistinguishable from a bad day by
   price alone, so those land in the review list instead of being adjusted. Confirm one and add it to
   `MANUAL_RATIOS`. A corporate-actions feed is the only complete fix, and that is a data licence
   rather than an algorithm.
2. **Dividends** are never in the price at all, so ex-dividend days leave small downward steps.
   Immaterial for a 14-bar RSI, material for long-horizon backtests.
3. **Symbol changes** — the key is `(exchange, symbol)`. A renamed ticker starts a fresh history and
   warms up again. Keying on ISIN would survive renames but not ISIN changes; neither is free.
4. **Untraded sessions** — an illiquid symbol's zero-volume days sit in the series as flat bars.
   They depress ATR and distort RVOL. Filter on `turnover` before trusting anything, as the example
   screen does.
5. **Revised files** — the cache keeps whatever was first downloaded. Delete a day's CSV to re-pull it.
6. **UDiFF starts mid-2024**, so `HISTORY_DAYS` beyond that silently yields nothing for the older
   dates; the old bhavcopy format needs its own parser.